<a href="https://colab.research.google.com/github/siddharthsinh-dev/iu-bsc-thesis-llm-comparison/blob/main/notebooks/exp2_c_mistral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Thesis**: A Comparative Study of Large Language Models for Financial Sentiment Analysis and Their Predictive Potential for Short-Term Stock Price Movement

**Component: Experiment 2** - Model C: Mistral (7B Instruct v0.3)

**Description:** This notebook loads the prepared 1000 headline dataset and evaluates Mistral 7B on Experiment 2 using zero-shot prompting. It maps sentiment predictions to directional signals and computes directional accuracy, precision, recall, and F1-score based on next-day stock price movement.

Select T4 GPU as runtime.

In [ ]:
# Install bitsandbytes — restart required after this

!pip install -q -U bitsandbytes accelerate

Go to runtime -> restart this session again -> then run cell 1 and run cell 2

In [ ]:
# Import required libraries — Experiment 2c: Mistral

import pandas as pd
import numpy as np
import torch
import warnings
import gc

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)

warnings.filterwarnings("ignore")

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU detected    : {torch.cuda.get_device_name(0)}")

Add the API Key from Hugging Face using "Add New Secret" in Google Colab

In [ ]:
# Connect to Hugging Face and load Experiment 2 dataset

from google.colab import userdata, drive
from huggingface_hub import login

drive.mount("/content/drive")

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

df_exp2 = pd.read_csv("/content/drive/MyDrive/Thesis_Data/exp2_dataset.csv")

print("Dataset loaded.")
print(f"Total headlines : {len(df_exp2)}")
print(f"\nMovement distribution:")
print(df_exp2["movement"].value_counts())

In [ ]:
# Load Mistral 7B Instruct v0.3

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading Mistral 7B Instruct v0.3...")

mistral_tokenizer = AutoTokenizer.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    token=hf_token
)

mistral_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

print("Mistral 7B loaded successfully.")

In [ ]:
# Define Mistral classifier — positive or negative only

import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

def classify_mistral(text):
    prompt = f"""You are a financial sentiment classifier.

Classify the sentiment of this financial text as exactly one word: positive or negative.

Text: {text}

Respond with only one word: positive or negative.

Sentiment:"""

    inputs = mistral_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(mistral_model.device)

    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=mistral_tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]
    generated = mistral_tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    ).strip().lower()

    if "negative" in generated:
        return "negative"
    else:
        return "positive"

print("Mistral classifier ready.")

In [ ]:
# Run Mistral on 1,000 headlines dataset

headlines = df_exp2["headline"].tolist()
mistral_preds = []
total = len(headlines)

print(f"Running Mistral on {total} headlines...")
print("-" * 40)

for i, text in enumerate(headlines):
    label = classify_mistral(text)
    mistral_preds.append(label)

    if (i + 1) % 100 == 0:
        print(f"  Progress: {i+1}/{total}")

print(f"\nDone. Total predictions: {len(mistral_preds)}")
print(f"\nSentiment distribution:")
print(pd.Series(mistral_preds).value_counts())

In [ ]:
# Map sentiment to directional prediction and evaluate

df_exp2["mistral_sentiment"] = mistral_preds
df_exp2["mistral_direction"] = df_exp2["mistral_sentiment"].map({
    "positive": 1,
    "negative": 0
})

true_movement = df_exp2["movement"].tolist()
pred_movement = df_exp2["mistral_direction"].tolist()

acc  = accuracy_score(true_movement, pred_movement)
prec = precision_score(true_movement, pred_movement, average="macro")
rec  = recall_score(true_movement, pred_movement, average="macro")
f1   = f1_score(true_movement, pred_movement, average="macro")

print("=" * 50)
print("Mistral 7B — Experiment 2 Results (next-day movement)")
print("=" * 50)
print(f"  Directional Accuracy : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision            : {prec:.4f}")
print(f"  Recall               : {rec:.4f}")
print(f"  F1-Score             : {f1:.4f}")
print("=" * 50)
print(classification_report(true_movement, pred_movement,
      target_names=["Down (0)", "Up (1)"]))

In [ ]:
# Save Mistral Experiment 2 results

mistral_exp2_scores = {
    "model": "Mistral 7B",
    "directional_accuracy": round(acc, 4),
    "precision": round(prec, 4),
    "recall": round(rec, 4),
    "f1_score": round(f1, 4)
}

print("Mistral 7B — Experiment 2 Results Summary")
print(pd.DataFrame([mistral_exp2_scores]))

df_exp2.to_csv("/content/drive/MyDrive/Thesis_Data/exp2_mistral_preds.csv", index=False)
print("\nSaved to Google Drive.")